# GMaster vs NaMaster, live on a hosted GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/licongxu/GMaster/blob/cursor/kaggle-demo-1a91/examples/gmaster_colab_demo.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/licongxu/GMaster/blob/cursor/kaggle-demo-1a91/examples/gmaster_colab_demo.ipynb)

Same map, same mask, same estimator settings, two codes:

* **NaMaster** (`pymaster`, CPU, the reference) and
* **GMaster** (JAX + the float32 difference-form CUDA march, GPU),

each run as `NmtField(n_iter=3)` + `NmtWorkspace.compute_coupling_matrix` + `compute_coupled_cell` + `decouple_cell`.
The notebook reports, side by side, **wall-clock**, **peak memory** (host RSS for both, device for GMaster) and the
**bandpower ratio** `C_ell^GM / C_ell^NM - 1`. Nothing is loaded from a cache: every number is produced in this session.

**Runtime.** Colab: `Runtime > Change runtime type > GPU`. Kaggle: session **Accelerator = GPU T4 x2** (or GPU P100) and **Internet on**, then Run All. Free Colab is usually one T4 (15 GB, 2 vCPU). Kaggle's free GPU is a T4 pair (16 GB each, more host RAM/CPU) or a P100 (16 GB). The demo pins `CUDA_VISIBLE_DEVICES=0` so a T4 pair still times one GPU. Default `NSIDE = 1024` takes a few minutes. The notebook prints the device it got. A TPU or CPU runtime runs GMaster without the CUDA march (correctness only, no timing claim).

**Choose the input** in cell 2: `NSIDE`, `SPIN` (0: TT; 2: EE, EB, BE, BB), the mask, and a synthetic CAMB + white-noise
realisation or your own HEALPix FITS. No option changes the estimator: both codes always get `n_iter = 3`,
`lmax = 3 NSIDE - 1`, `NmtBin.from_lmax_linear(lmax, 50)` and the same mask array.

Reference numbers from the shipped real-map demos (RTX PRO 6000 Blackwell vs 192 CPU cores, `NSIDE = 4096`):
ACT DR6 TT 7.63 s vs 70.4 s, rms ratio 1.4e-5; FLAMINGO Compton-y 8.64 s vs 69.5 s, rms 2.3e-7
(`examples/act_dr6_tt_example`, `examples/flamingo_y_tt_example`, and the Letter under `paper/gmaster_letter`).


In [ ]:
#@title 1. Install (Colab / Kaggle; ~5 min, `pymaster` and `fitsio` build from source) { display-mode: "form" }
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_KAGGLE = os.path.isdir("/kaggle/working") or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
IN_HOSTED = IN_COLAB or IN_KAGGLE

# One GPU for the 1-1 timing. Kaggle "GPU T4 x2" otherwise exposes two devices
# and the default jax calculator shards scalar SHTs at L>=2048.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# This branch: act-dr6-demo does not contain this notebook or the s2fft pin.
GMASTER_REF = "cursor/kaggle-demo-1a91"  #@param {type:"string"}
# PyPI s2fft 1.4.0 does `from s2fft_lib import _s2fft` at import and crashes on
# Colab Python 3.13 (the extension is a namespace without `_s2fft`). This SHA
# has the optional import (#362) and `_ftm_flm_primitive` (#380).
S2FFT_GIT = "git+https://github.com/astro-informatics/s2fft.git@86a4dff303539202ebbfa28661d85331e0e2bff4"


def sh(cmd, extra_env=None):
    print("$", cmd, flush=True)
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=env)
    if r.returncode:
        print(r.stdout[-4000:], r.stderr[-4000:])
        raise RuntimeError(f"failed: {cmd}")


if IN_HOSTED:
    sh("apt-get update -qq > /dev/null && "
       "apt-get install -y -qq autoconf automake libtool libgsl-dev libfftw3-dev libcfitsio-dev > /dev/null")
    print("pymaster builds from source here (~3 min) ...", flush=True)
    sh(f"{sys.executable} -m pip install -q camb psutil healpy pymaster")
    subprocess.run(f"{sys.executable} -m pip uninstall -y s2fft s2fft-lib s2fft_lib",
                   shell=True, capture_output=True)
    print("installing s2fft from git, CPU extension only (host nvcc would try extra sm_XX) ...", flush=True)
    # Hide nvcc so CMake takes the NO_CUDA_COMPILER branch and still produces s2fft_lib._s2fft.
    sh(f"{sys.executable} -m pip install -q --upgrade --no-cache-dir {S2FFT_GIT}",
       extra_env={"CMAKE_CUDA_COMPILER": "/does/not/exist"})
    sh(f"{sys.executable} -m pip install -q --upgrade --no-cache-dir git+https://github.com/licongxu/GMaster.git@{GMASTER_REF}")
    import importlib
    from importlib.metadata import version
    for k in list(sys.modules):
        if k == "s2fft" or k.startswith("s2fft.") or k.startswith("s2fft_lib"):
            del sys.modules[k]
    for mod in ("s2fft", "s2fft.utils.healpix_ffts", "s2fft.transforms._ftm_flm_primitive", "gmaster"):
        importlib.import_module(mod)
        print("imported", mod)
    print({p: version(p) for p in ("jax", "pymaster", "gmaster", "s2fft", "healpy", "camb")})
    print("host:", "Kaggle" if IN_KAGGLE else "Colab")
    print("If a previous cell already imported s2fft 1.4.0: restart the session, then run from cell 2.")
else:
    print("Not in Colab/Kaggle: assuming gmaster, pymaster, camb, healpy, psutil are already installed.")



In [ ]:
#@title 2. Choose the input { display-mode: "form" }
import os

NSIDE = 1024  #@param [512, 1024, 2048, 4096] {type:"raw"}
SPIN = 0  #@param [0, 2] {type:"raw"}
MASK = "galactic cut + C1 apodisation"  #@param ["full-sky", "galactic cut", "galactic cut + C1 apodisation"]
MAP_SOURCE = "synthetic CAMB realisation"  #@param ["synthetic CAMB realisation", "HEALPix FITS (upload or path)"]
FITS_PATH = ""  #@param {type:"string"}
SEED = 1234  #@param {type:"integer"}

# Estimator settings shared by both codes; not options.
N_ITER = 3            # NaMaster's default, passed explicitly to both NmtField calls
NLB = 50              # NmtBin.from_lmax_linear(LMAX, NLB) for both
NOISE_UK_ARCMIN = 10.0  # white noise added to the synthetic map so the high-ell ratio is not 0/0
GAL_CUT_DEG = 20.0
APO_DEG = 2.0

# Headless runs (nbconvert / papermill) override the form through the environment.
NSIDE = int(os.environ.get("GM_DEMO_NSIDE", NSIDE))
SPIN = int(os.environ.get("GM_DEMO_SPIN", SPIN))
MASK = os.environ.get("GM_DEMO_MASK", MASK)
MAP_SOURCE = os.environ.get("GM_DEMO_MAP_SOURCE", MAP_SOURCE)
FITS_PATH = os.environ.get("GM_DEMO_FITS_PATH", FITS_PATH)

assert SPIN in (0, 2), SPIN
LMAX = 3 * NSIDE - 1
NPIX = 12 * NSIDE * NSIDE
NMAPS = 1 if SPIN == 0 else 2
NCLS = 1 if SPIN == 0 else 4
print(f"NSIDE={NSIDE}  LMAX={LMAX}  SPIN={SPIN}  MASK={MASK!r}  MAP_SOURCE={MAP_SOURCE!r}  n_iter={N_ITER}  nlb={NLB}")

In [ ]:
#@title 3. Runtime report and memory guard { display-mode: "form" }
import os
import shutil
import subprocess
import time

import psutil

# Before jax is imported: no 75% pre-allocation, so the device peak below is what GMaster used.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
# Kaggle GPU T4 x2 would otherwise expose two devices; time one GPU vs CPU NaMaster.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import gmaster as gm
import pymaster as nmt
from gmaster import _march_v2
from gmaster import _coupling_tt_cuda
from gmaster._cuda_gpu import is_cuda_device_kind, on_cuda_gpu
from gmaster._nmt_bin64 import mcm_needs_i64, patch_pymaster
from gmaster._spin_march_pallas import fold_requested, march_requested
from gmaster.utils import _use_pallas_sht

dev = jax.devices()[0]
PLATFORM = dev.platform
DEVICE_KIND = dev.device_kind
NCORES = len(os.sched_getaffinity(0))
HOST_LIMIT = psutil.virtual_memory().total
HOST_GB = HOST_LIMIT / 1e9
nvcc = os.environ.get("GMASTER_NVCC") or shutil.which("nvcc") or (
    "/usr/local/cuda/bin/nvcc" if os.path.exists("/usr/local/cuda/bin/nvcc") else None)

print(f"JAX {jax.__version__}  backend={PLATFORM}  device={DEVICE_KIND}")
print(f"CPU cores visible to this process: {NCORES}   host RAM: {HOST_GB:.1f} GB   nvcc: {nvcc}")

DEV_LIMIT = None
if PLATFORM == "gpu":
    stats = dev.memory_stats() or {}
    DEV_LIMIT = stats.get("bytes_limit") or stats.get("bytes_reservable_limit")
    if not DEV_LIMIT:
        q = subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True)
        DEV_LIMIT = float(q.stdout.split()[0]) * 1024 ** 2 if q.returncode == 0 else None
    print(f"device memory: {DEV_LIMIT / 1e9:.1f} GB" if DEV_LIMIT else "device memory: unknown")
    print("building the v2 march and TT coupling kernels with nvcc (first time only, cached in ~/.cache/gmaster) ...", flush=True)
    print(f"device_kind={DEVICE_KIND!r}  cuda_gpu={on_cuda_gpu()}  "
          f"kind_ok={is_cuda_device_kind(DEVICE_KIND)}")
    MARCH_LIB = _march_v2.enabled(LMAX + 1)
    print("v2 CUDA march library:", "enabled" if MARCH_LIB else f"UNAVAILABLE: {_march_v2.unavailable_reason()}")
    if SPIN == 0:
        MARCH = bool(MARCH_LIB and _use_pallas_sht(LMAX + 1, 0) and fold_requested(NSIDE, LMAX + 1))
        print(f"SHT route: {'v2 CUDA march' if MARCH else 'NOT the CUDA march'}  "
              f"pallas={_use_pallas_sht(LMAX + 1, 0)}  fold={fold_requested(NSIDE, LMAX + 1)}")
    else:
        MARCH = bool(MARCH_LIB and march_requested(SPIN, L=LMAX + 1, nside=NSIDE))
        print(f"SHT route: {'v2 CUDA march' if MARCH else 'NOT the CUDA march'}  "
              f"march_requested={march_requested(SPIN, L=LMAX + 1, nside=NSIDE)}")
    if not MARCH:
        raise RuntimeError(
            f"v2 CUDA march is not serving the SHT on {DEVICE_KIND!r}. "
            "Refusing to time the generic s2fft fallback.")
    print("CUDA TT coupling:", "enabled" if _coupling_tt_cuda.enabled()
          else f"UNAVAILABLE: {_coupling_tt_cuda.unavailable_reason()}")
    from gmaster.workspaces import _TT_QUADRATURE_LMAX
    if LMAX >= _TT_QUADRATURE_LMAX:
        print(f"TT coupling route: Gauss-Legendre GEMM (lmax={LMAX} >= {_TT_QUADRATURE_LMAX})")
    else:
        print(f"TT coupling route: threej recurrence (lmax={LMAX} < {_TT_QUADRATURE_LMAX})")
    a = jax.device_put(jnp.ones((2048, 2048), dtype=jnp.float32))
    t0 = time.perf_counter(); jax.block_until_ready(a @ a); cold = time.perf_counter() - t0
    t0 = time.perf_counter(); jax.block_until_ready(a @ a); warm = time.perf_counter() - t0
    print(f"GPU sanity: 2048x2048 f32 matmul cold {cold:.3f}s warm {warm:.4f}s on {a.device}")
    del a
else:
    MARCH = False
    print("No GPU: GMaster runs on", PLATFORM, "without the CUDA march. Correctness check only; no timing claim.")

# Rough size estimates (bytes) so an impossible choice fails here, not in an OOM mid-run.
nls = LMAX + 1
mcm_bytes = 8 * NCLS ** 2 * nls ** 2                     # unbinned mode-coupling matrix, float64
est_host = 8 * NPIX * (NMAPS + 4) + 2 * mcm_bytes          # maps + synfast temporaries + NaMaster's MCM and inverse
est_dev = mcm_bytes + 8 * NPIX * (NMAPS + 2) + 8 * NMAPS * nls * nls + 2.5e9   # MCM + maps/mask + alms + march buffers
print(f"estimated host RAM need ~{est_host / 1e9:.1f} GB (have {HOST_GB:.1f});"
      + (f"  estimated device need ~{est_dev / 1e9:.1f} GB (have {DEV_LIMIT / 1e9:.1f})" if DEV_LIMIT else ""))
if est_host > HOST_LIMIT:
    raise MemoryError(f"NSIDE={NSIDE} SPIN={SPIN} needs ~{est_host / 1e9:.0f} GB host RAM for NaMaster's "
                      f"{NCLS * nls}x{NCLS * nls} coupling matrix; pick a High-RAM runtime or a smaller NSIDE.")
if DEV_LIMIT and est_dev > DEV_LIMIT:
    raise MemoryError(f"NSIDE={NSIDE} SPIN={SPIN} needs ~{est_dev / 1e9:.0f} GB on the GPU; "
                      f"pick a larger GPU or a smaller NSIDE.")
if DEV_LIMIT and est_dev > 0.7 * DEV_LIMIT:
    print("Device estimate is within 30% of the card: this will be tight.")

# NaMaster indexes its flattened MCM with int32 and segfaults at 4096 spin 2; gmaster carries a 64-bit binning
# fallback that hooks a Python-side NmtBin._bin_mcm, which the PyPI pymaster 3.0.1 (C-side binning) does not have.
if mcm_needs_i64(nls, NCLS):
    try:
        patch_pymaster()
    except AttributeError as exc:
        raise RuntimeError(
            f"NSIDE={NSIDE} SPIN={SPIN}: this pymaster build bins the coupling matrix in C with int32 indices and "
            "segfaults at this size. Install pymaster from https://github.com/LSSTDESC/NaMaster (main) or pick "
            "NSIDE <= 2048 for spin 2.") from exc
    print("NaMaster's flattened MCM index overflows int32 at this size; gmaster's 64-bit binning fallback is installed.")


In [ ]:
#@title 4. Map and mask (the same numpy arrays go to both estimators) { display-mode: "form" }
import time

import camb
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pymaster as nmt

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

rng = np.random.default_rng(SEED)
t0 = time.perf_counter()

if MAP_SOURCE == "synthetic CAMB realisation":
    pars = camb.set_params(H0=67.36, ombh2=0.02237, omch2=0.1200, mnu=0.06, omk=0, tau=0.0544,
                           As=2.1e-9, ns=0.9649, lmax=LMAX, lens_potential_accuracy=1)
    cl = camb.get_results(pars).get_cmb_power_spectra(pars, CMB_unit="muK", raw_cl=True, lmax=LMAX)["total"]
    cl_tt, cl_ee, cl_bb, cl_te = cl.T
    pix_arcmin2 = hp.nside2pixarea(NSIDE, degrees=True) * 3600.0
    sigma_pix = NOISE_UK_ARCMIN / np.sqrt(pix_arcmin2)
    if SPIN == 0:
        t = hp.synfast(cl_tt, NSIDE, lmax=LMAX, new=True)
        maps = [t + sigma_pix * rng.standard_normal(NPIX)]
    else:
        t, q, u = hp.synfast([cl_tt, cl_ee, cl_bb, cl_te], NSIDE, lmax=LMAX, new=True, pol=True)
        maps = [q + np.sqrt(2) * sigma_pix * rng.standard_normal(NPIX),
                u + np.sqrt(2) * sigma_pix * rng.standard_normal(NPIX)]
    source = f"CAMB Planck-2018-like lensed spectrum + {NOISE_UK_ARCMIN:g} uK-arcmin white noise, seed {SEED}"
else:
    if FITS_PATH:
        fname = FITS_PATH
    elif colab_files is not None:
        fname = next(iter(colab_files.upload()))
    else:
        raise RuntimeError("Set FITS_PATH in cell 2 (Colab upload is unavailable here).")
    fields = 0 if SPIN == 0 else (1, 2)
    m = np.atleast_2d(hp.read_map(fname, field=fields, dtype=np.float64))
    if hp.get_nside(m[0]) != NSIDE:
        m = np.array([hp.ud_grade(x, NSIDE) for x in m])
    maps = [np.asarray(x, dtype=np.float64) for x in m]
    source = f"{fname} (fields {fields}, ud_grade to {NSIDE})"

theta, _ = hp.pix2ang(NSIDE, np.arange(NPIX))
b_deg = 90.0 - np.degrees(theta)
if MASK == "full-sky":
    mask = np.ones(NPIX)
else:
    mask = (np.abs(b_deg) > GAL_CUT_DEG).astype(np.float64)
    if MASK.endswith("apodisation"):
        mask = nmt.mask_apodization(mask, APO_DEG, apotype="C1")
mask = np.asarray(mask, dtype=np.float64)
FSKY = float(np.mean(mask > 0))
print(f"map: {source}\nmask: {MASK}, f_sky = {FSKY:.3f}\nprepared in {time.perf_counter() - t0:.1f} s (not timed below)")

vmax = float(np.percentile(np.abs(maps[0][mask > 0]), 99))
hp.mollview(np.where(mask > 0, maps[0], hp.UNSEEN), title=("T" if SPIN == 0 else "Q") + " x mask",
            unit="uK", min=-vmax, max=vmax, cmap="RdBu_r")
plt.show()


In [ ]:
#@title 5. Timing and memory helpers { display-mode: "form" }
import contextlib
import shutil
import subprocess
import threading
import time

import psutil

SMI = shutil.which("nvidia-smi") if PLATFORM == "gpu" else None


def smi_used_bytes():
    """GPU memory in use on device 0 as nvidia-smi reports it (context + XLA pool), or None."""
    if SMI is None:
        return None
    q = subprocess.run([SMI, "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"],
                       capture_output=True, text=True)
    return float(q.stdout.split()[0]) * 1024 ** 2 if q.returncode == 0 and q.stdout.strip() else None


@contextlib.contextmanager
def measured(record):
    """Wall-clock and peak host RSS (200 ms samples). nvidia-smi is sampled only at
    start/end: querying it in the loop serializes the CUDA context on Colab T4."""
    proc = psutil.Process()
    stop = threading.Event()
    start_rss = proc.memory_info().rss
    peak = [start_rss]
    dev_peak = [smi_used_bytes() or 0.0]

    def sample():
        while not stop.is_set():
            peak[0] = max(peak[0], proc.memory_info().rss)
            stop.wait(0.2)

    th = threading.Thread(target=sample, daemon=True)
    th.start()
    t0 = time.perf_counter()
    try:
        yield
    finally:
        record["t"] = time.perf_counter() - t0
        stop.set()
        th.join()
        peak[0] = max(peak[0], proc.memory_info().rss)
        if SMI:
            dev_peak[0] = max(dev_peak[0], smi_used_bytes() or 0.0)
        record["rss_start"] = start_rss
        record["rss_peak"] = peak[0]
        record["dev_peak_smi"] = dev_peak[0] if SMI else None


def gb(x):
    return f"{x / 1e9:.2f} GB"


print("ready")

In [ ]:
#@title 6. GMaster (GPU): run twice, the first includes JIT compilation { display-mode: "form" }
import time

bins_gm = gm.NmtBin.from_lmax_linear(LMAX, NLB)
ell = np.asarray(bins_gm.get_effective_ells())


def gmaster_once():
    stages = {}
    t0 = time.perf_counter()
    f = gm.NmtField(mask, maps, n_iter=N_ITER)
    jax.block_until_ready(f.get_alms())
    stages["field"] = time.perf_counter() - t0
    t1 = time.perf_counter()
    ws = gm.NmtWorkspace()
    ws.compute_coupling_matrix(f, f, bins_gm)
    jax.block_until_ready(ws.mcm_binned)
    stages["coupling"] = time.perf_counter() - t1
    t2 = time.perf_counter()
    pcl = gm.compute_coupled_cell(f, f)
    jax.block_until_ready(pcl)
    stages["coupled_cell"] = time.perf_counter() - t2
    t3 = time.perf_counter()
    cl = np.asarray(jax.block_until_ready(ws.decouple_cell(pcl)))
    stages["decouple"] = time.perf_counter() - t3
    return cl, stages


gm_runs = []
for r in range(2):
    rec = {}
    with measured(rec):
        cl_gm, rec["stages"] = gmaster_once()
    gm_runs.append(rec)
    stages = rec["stages"]
    print(f"GMaster run {r}: {rec['t']:.2f} s   host RSS peak {gb(rec['rss_peak'])} "
          f"(+{gb(rec['rss_peak'] - rec['rss_start'])} during the run)"
          + ("   [includes JIT compilation]" if r == 0 else ""))
    print("  stages: field {field:.2f}s  coupling {coupling:.2f}s  "
          "coupled_cell {coupled_cell:.2f}s  decouple {decouple:.2f}s".format(**stages),
          flush=True)

GM = gm_runs[-1]
GM_DEV_PEAK = (dev.memory_stats() or {}).get("peak_bytes_in_use") if PLATFORM == "gpu" else None
if PLATFORM == "gpu":
    print(f"GMaster device peak on {DEVICE_KIND}: XLA peak_bytes_in_use "
          f"{gb(GM_DEV_PEAK) if GM_DEV_PEAK else 'n/a'} (since process start); "
          f"nvidia-smi memory.used peak {gb(max(r['dev_peak_smi'] or 0 for r in gm_runs))}"
          + (f" of {gb(DEV_LIMIT)}" if DEV_LIMIT else ""))
print("decoupled C_ell shape:", cl_gm.shape)

In [ ]:
#@title 7. NaMaster (CPU): the same estimator, once { display-mode: "form" }
bins_nm = nmt.NmtBin.from_lmax_linear(LMAX, NLB)
assert np.allclose(np.asarray(bins_nm.get_effective_ells()), ell)

NM = {}
with measured(NM):
    f_nm = nmt.NmtField(mask, maps, n_iter=N_ITER)
    ws_nm = nmt.NmtWorkspace()
    ws_nm.compute_coupling_matrix(f_nm, f_nm, bins_nm)
    cl_nm = ws_nm.decouple_cell(nmt.compute_coupled_cell(f_nm, f_nm))
cl_nm = np.asarray(cl_nm)
print(f"NaMaster: {NM['t']:.2f} s on {NCORES} cores   host RSS peak {gb(NM['rss_peak'])} "
      f"(+{gb(NM['rss_peak'] - NM['rss_start'])} during the run)")
del f_nm, ws_nm

In [ ]:
#@title 8. Overlay, ratio, and the side-by-side table { display-mode: "form" }
from IPython.display import Markdown, display

BLUE, GREY, BLACK = "#0072B2", "#666666", "#000000"
components = [("TT", 0)] if SPIN == 0 else [("EE", 0), ("BB", 3)]
dl = ell * (ell + 1) / (2 * np.pi)

fig, axes = plt.subplots(2, len(components), sharex=True, figsize=(7.2 * len(components), 4.8), squeeze=False,
                         gridspec_kw={"height_ratios": [2.2, 1]})
stats = {}
for j, (name, k) in enumerate(components):
    ratio = cl_gm[k] / cl_nm[k] - 1.0
    stats[name] = (float(np.std(ratio)), float(np.max(np.abs(ratio))))
    ax0, ax1 = axes[0, j], axes[1, j]
    ax0.plot(ell, dl * cl_gm[k], color=BLUE, lw=1.4, label=f"GMaster ({DEVICE_KIND}) {GM['t']:.2f} s")
    ax0.plot(ell, dl * cl_nm[k], color=GREY, lw=1.1, ls="--", label=f"NaMaster ({NCORES} cores) {NM['t']:.1f} s")
    ax0.set_yscale("log")
    ax0.set_ylabel(r"$D_\ell=\ell(\ell+1)C_\ell/2\pi$")
    ax0.set_title(f"{name}, NSIDE={NSIDE}, f_sky={FSKY:.2f}")
    ax0.legend(frameon=False)
    ax1.axhline(0, color=BLACK, lw=0.6)
    ax1.plot(ell, 1e5 * ratio, color=BLUE, lw=0.9)
    ax1.set_ylabel(r"$(C_\ell^{\rm GM}/C_\ell^{\rm NM}-1)\times10^{5}$")
    ax1.set_xlabel(r"$\ell$")
fig.tight_layout()
plt.show()

rows = [
    ("wall-clock (field + coupling matrix + coupled cell + decouple)", f"{GM['t']:.2f} s", f"{NM['t']:.2f} s"),
    ("first GMaster run incl. JIT", f"{gm_runs[0]['t']:.2f} s", "-"),
    ("GMaster field (warmed)", f"{GM['stages']['field']:.2f} s", "-"),
    ("GMaster coupling matrix (warmed)", f"{GM['stages']['coupling']:.2f} s", "-"),
    ("GMaster coupled cell (warmed)", f"{GM['stages']['coupled_cell']:.2f} s", "-"),
    ("GMaster decouple (warmed)", f"{GM['stages']['decouple']:.2f} s", "-"),
    ("hardware", DEVICE_KIND if PLATFORM == "gpu" else PLATFORM, f"{NCORES} CPU cores"),
    ("host RSS peak during run", gb(GM["rss_peak"]), gb(NM["rss_peak"])),
    ("host RSS increase during run", gb(GM["rss_peak"] - GM["rss_start"]), gb(NM["rss_peak"] - NM["rss_start"])),
    ("device peak, XLA peak_bytes_in_use (GMaster only)", gb(GM_DEV_PEAK) if GM_DEV_PEAK else "n/a", "-"),
    ("device peak, nvidia-smi memory.used during run", gb(GM["dev_peak_smi"]) if GM["dev_peak_smi"] else "n/a",
     gb(NM["dev_peak_smi"]) if NM["dev_peak_smi"] else "-"),
]
md = [f"### NSIDE={NSIDE}, lmax={LMAX}, spin={SPIN}, n_iter={N_ITER}, nlb={NLB}, mask: {MASK} (f_sky={FSKY:.3f})",
      "", "| | GMaster | NaMaster |", "|---|---|---|"]
md += [f"| {a} | {b} | {c} |" for a, b, c in rows]
md += ["", f"**NaMaster / GMaster wall-clock: {NM['t'] / GM['t']:.3g}x**", ""]
md += [f"**{name}: rms(GM/NM-1) = {s[0]:.2e}, max|GM/NM-1| = {s[1]:.2e}**  " for name, s in stats.items()]
if not MARCH:
    md += ["", "The v2 CUDA march was not available in this runtime, so the GMaster column is not the GPU route "
           "the shipped numbers describe."]
faster = GM["t"] < NM["t"]
accurate = all(s[0] < 1e-4 for s in stats.values())
dev_ok = (GM_DEV_PEAK is None) or (GM_DEV_PEAK < 15e9)
passed = bool(faster and accurate and MARCH and dev_ok)
md += ["", f"**demo test: {'PASS' if passed else 'FAIL'}** "
       f"(GMaster faster: {faster}; rms < 1e-4: {accurate}; march: {bool(MARCH)}; "
       f"device peak < 15 GB: {dev_ok})"]
display(Markdown("\n".join(md)))
if PLATFORM == "gpu" and not passed:
    raise RuntimeError("Hosted GPU demo test failed: GMaster must be faster than NaMaster with rms < 1e-4, "
                       "the v2 CUDA march enabled, and device peak under 15 GB.")


## What you are looking at

* Both columns come from the same `mask` and `maps` arrays, the same `lmax`, the same bins, and `n_iter = 3`; the
  only difference is the code. GMaster's default route is the float32 difference-form march (`gmaster/_march_v2.py`,
  `gmaster/_cuda/march_v2.cu`), which is why the ratio is at the 1e-5 level rather than 1e-13; the exact fp64 route
  is `GMASTER_MARCH_V2=0`.
* The GMaster time is the second run. The first includes XLA compilation (and, in cell 3, the one-off `nvcc` build
  of the v2 march). Scalar TT coupling at the default NSIDE uses a Gauss-Legendre GEMM (`P_l` table + two matmuls),
  not the threej offset scan. Cell 6 also prints field / coupling / coupled-cell / decouple splits so a slow kernel
  is visible.
* Host RSS is the whole notebook process; the "increase during run" row is the part each estimator added. Two device
  numbers are given: XLA's `peak_bytes_in_use` (what GMaster's arrays occupied, since the process started, with
  pre-allocation disabled) and the `nvidia-smi memory.used` peak sampled during the run (CUDA context plus allocator
  pool, what you would see on the card).
* Hosted CPUs are few (2 on free Colab, typically 4 on Kaggle GPU), so the NaMaster column is slower here than on a
  workstation. The shipped 4096 comparisons against 192 cores are in `examples/act_dr6_tt_example` and
  `examples/flamingo_y_tt_example`.
* To rerun with another `NSIDE`, `SPIN`, mask, or your own FITS: change cell 2 and run cells 2 to 8 again.
